# Data Prep & Transformation | Aula 3 · Prática
## Parte 1: o pedido como documento (continuação da Aula 2) · Parte 2: modelagem analítica em estrela

**Luciano Gillieron Gavinho, PhD.** · MBA em Engenharia de Dados · Universidade Presbiteriana Mackenzie

Este notebook reúne tudo o que vamos fazer hoje, para ninguém precisar abrir outro arquivo.

- **Parte 1** retoma o notebook da Aula 2 exatamente de onde paramos: o exemplo A.5 e os exercícios 4, 4b e 5, com a **mesma numeração** da Aula 2.
- **Parte 2** é nova: vamos transformar as tabelas transacionais da Olist num **modelo estrela** (uma tabela fato ligada a dimensões), o formato que a área analítica usa para responder perguntas de negócio com rapidez.

**Como trabalhar:** nas equipes de sempre, misturando quem programa e quem é de negócio. Quem não programa pode, e deve, usar IA para entender e adaptar o código. O importante é olhar os resultados e entender o que eles dizem.

### Termos usados neste notebook

| Termo | O que significa |
|---|---|
| **JSON** | *JavaScript Object Notation*: texto que guarda dados aninhados, com a estrutura junto do dado. |
| **Documento** | Registro autocontido em JSON, que pode **embutir** dados relacionados (os itens dentro do pedido). |
| **NoSQL** | Família de bancos não relacionais: documento, chave-valor, colunar e grafo. |
| **Modelagem dimensional** | Forma de organizar dados para análise em **fatos** (o que se mede) e **dimensões** (o contexto da medida). |
| **Tabela fato** | Tabela com os eventos medidos, como cada item vendido, e suas medidas numéricas (preço, frete). |
| **Dimensão** | Tabela de contexto que descreve o fato: produto, vendedor, cliente, tempo. |
| **Grão** | O que exatamente uma linha da tabela fato representa. É a primeira decisão do modelo. |
| **Medida** | Valor numérico da fato que se soma, conta ou faz média: preço, frete. |
| **Chave substituta** (SK, *surrogate key*) | Número inteiro criado pelo modelo para identificar cada linha de uma dimensão (1, 2, 3...). |
| **Estrela** (*star schema*) | Uma fato no centro ligada diretamente a cada dimensão. |
| **Snowflake** | Estrela em que uma dimensão é normalizada em subtabelas (o produto aponta para uma tabela de categoria). |
| **Dimensão degenerada** | Identificador que fica na própria fato, sem tabela de dimensão, como o `order_id`. |
| **SCD** | *Slowly Changing Dimension*: dimensão cujos atributos mudam com o tempo, como o endereço do cliente. |
| `ROW_NUMBER() OVER (ORDER BY x)` | Função SQL que numera as linhas 1, 2, 3... na ordem de `x`. Usada para gerar a chave substituta. |
| `generate_series(início, fim, intervalo)` | Função do DuckDB que gera uma sequência, por exemplo todas as datas de um período. |
| `COALESCE(a, b)` | Devolve `a`; se `a` for nulo, devolve `b`. Usada para trocar nulos por um valor padrão. |
| **Wide table** | Tabela larga: fato e atributos das dimensões já juntados numa tabela só, sem JOIN na consulta. Também chamada de OBT (*One Big Table*). |
| **Flat table** | Tabela plana, sem aninhamento nem relacionamentos, geralmente entregue como um arquivo único (CSV, Parquet) para outra ferramenta. |
| **Data Vault** | Modelo de integração com **hubs** (chaves de negócio), **links** (relacionamentos entre hubs) e **satélites** (atributos com histórico). |
| `md5(texto)` | Função que calcula um *hash*: um código de 32 caracteres derivado do texto. Usada no Data Vault como chave dos hubs e links. |

## 0 · Preparação do ambiente
Execute as duas células abaixo, como nas aulas anteriores. A primeira baixa os dados da Olist e instala o DuckDB; a segunda registra cada arquivo como uma *view* e cria as funções `sql()` (consulta e devolve uma tabela) e `tentar()` (executa um comando e mostra o erro, se o banco recusar).

In [1]:
import os, subprocess
if not os.path.exists("work-at-olist-data"):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/olist/work-at-olist-data.git"], check=True)
subprocess.run(["pip", "install", "-q", "duckdb"], check=False)
print(sorted(os.listdir("work-at-olist-data/datasets")))

['olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_orders_dataset.csv', 'olist_products_dataset.csv', 'olist_sellers_dataset.csv', 'product_category_name_translation.csv']


In [2]:
import duckdb, pandas as pd, json
DATA = "work-at-olist-data/datasets"
con = duckdb.connect()

FONTES = {
    "customers":  "olist_customers_dataset.csv",
    "geolocation":"olist_geolocation_dataset.csv",
    "orders":     "olist_orders_dataset.csv",
    "items":      "olist_order_items_dataset.csv",
    "payments":   "olist_order_payments_dataset.csv",
    "reviews":    "olist_order_reviews_dataset.csv",
    "products":   "olist_products_dataset.csv",
    "sellers":    "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}
for nome, arq in FONTES.items():
    con.execute(f"CREATE OR REPLACE VIEW {nome} AS SELECT * FROM read_csv_auto('{DATA}/{arq}', header=true)")

def sql(q):
    """Executa uma consulta no DuckDB e devolve um DataFrame."""
    return con.sql(q).df()

def tentar(comando):
    """Executa um comando SQL e, se o banco recusar, mostra o erro em vez de interromper o notebook."""
    try:
        con.execute(comando)
        print("OK: comando executado.")
    except Exception as e:
        print("RECUSADO PELO BANCO:", type(e).__name__)
        print(str(e)[:300])

sql("SELECT COUNT(*) AS pedidos FROM orders")

,pedidos
0,99441


---
# Parte 1 · Continuação da Aula 2: o pedido como documento (NoSQL)
Na Aula 2 fizemos os exercícios 1 a 3 (modelo relacional). Agora seguimos com o exemplo A.5 e os exercícios 4, 4b e 5, **com a mesma numeração** do notebook da Aula 2.

### A.5 (da Aula 2) · De linhas para documento JSON, e de volta
`struct_pack(campo := valor, ...)` monta um registro com campos nomeados; `list(...)` junta várias linhas numa lista; `to_json(...)` converte para texto JSON. Para ler de volta, `json_extract_string(doc, '$.campo')` extrai um valor e `json_array_length(doc, '$.lista')` conta os elementos de uma lista.

Abaixo, um documento por estado com a lista de cidades dos vendedores daquele estado (limitado a 3 cidades, para caber na tela).

In [3]:
docs = sql("""
SELECT seller_state,
       to_json(struct_pack(
           uf      := seller_state,
           cidades := list(DISTINCT seller_city ORDER BY seller_city)[1:3]
       )) AS doc
FROM sellers
WHERE seller_state IN ('AM', 'MA')
GROUP BY seller_state
""")
display(docs)
print(json.dumps(json.loads(docs.loc[0, "doc"]), indent=2, ensure_ascii=False))

con.register("docs_exemplo", docs)   # registra o DataFrame como tabela para consultar em SQL
sql("""
SELECT json_extract_string(doc, '$.uf') AS uf,
       json_array_length(doc, '$.cidades') AS qtd_cidades
FROM docs_exemplo
""")

,seller_state,doc
0,MA,"{""uf"":""MA"",""cidades"":[""sao luis""]}"
1,AM,"{""uf"":""AM"",""cidades"":[""manaus""]}"


{
  "uf": "MA",
  "cidades": [
    "sao luis"
  ]
}


,uf,qtd_cidades
0,MA,1
1,AM,1


## 4 · O pedido como documento (NoSQL)
Num banco de documentos, a pergunta de modelagem muda: em vez de "como evitar redundância?", passa a ser "como a aplicação lê o dado?". Se a tela do pedido mostra cliente, itens e pagamentos juntos, faz sentido **embutir** tudo num só documento.
1. Crie a tabela `pedidos_doc` com uma linha por pedido e uma coluna `doc` em JSON contendo: dados do pedido, um objeto `cliente` e as listas `itens` e `pagamentos`.
2. Mostre o documento de um pedido com 3 itens, formatado.
3. Consultando **só os documentos**, calcule o total de itens e o faturamento (soma dos preços). Confira contra a tabela `items`.
4. Quantos documentos têm a lista de itens vazia ou nula? De onde vêm?
5. Qual o tamanho total dos documentos em MB? Compare com o tamanho dos CSVs de `orders`, `customers`, `items` e `payments` juntos.

In [4]:
con.execute("""
CREATE OR REPLACE TABLE pedidos_doc AS
SELECT o.order_id,
  to_json(struct_pack(
    order_id    := o.order_id,
    status      := o.order_status,
    comprado_em := o.order_purchase_timestamp,
    cliente     := struct_pack(customer_id := c.customer_id, pessoa := c.customer_unique_id,
                               cidade := c.customer_city, uf := c.customer_state),
    itens       := (SELECT list(struct_pack(item := i.order_item_id, produto := i.product_id,
                                            vendedor := i.seller_id, preco := i.price, frete := i.freight_value)
                                ORDER BY i.order_item_id)
                    FROM items i WHERE i.order_id = o.order_id),
    pagamentos  := (SELECT list(struct_pack(seq := p.payment_sequential, tipo := p.payment_type,
                                            parcelas := p.payment_installments, valor := p.payment_value)
                                ORDER BY p.payment_sequential)
                    FROM payments p WHERE p.order_id = o.order_id)
  )) AS doc
FROM orders o JOIN customers c USING (customer_id)
""")
print("documentos:", con.sql("SELECT COUNT(*) FROM pedidos_doc").fetchall()[0][0])

exemplo = con.sql("""
SELECT doc FROM pedidos_doc
WHERE json_array_length(doc, '$.itens') = 3
ORDER BY order_id LIMIT 1""").fetchall()[0][0]
print(json.dumps(json.loads(exemplo), indent=2, ensure_ascii=False))

display(sql("""
SELECT
  (SELECT SUM(json_array_length(doc, '$.itens')) FROM pedidos_doc) AS itens_nos_docs,
  (SELECT COUNT(*) FROM items)                                    AS itens_na_tabela,
  (SELECT ROUND(SUM(CAST(x.value ->> 'preco' AS DOUBLE)), 2)
     FROM pedidos_doc, json_each(doc, '$.itens') x)               AS faturamento_docs,
  (SELECT ROUND(SUM(price), 2) FROM items)                        AS faturamento_tabela,
  (SELECT COUNT(*) FROM pedidos_doc
     WHERE COALESCE(json_array_length(doc, '$.itens'), 0) = 0)    AS docs_sem_itens,
  (SELECT ROUND(SUM(LENGTH(doc)) / 1e6, 1) FROM pedidos_doc)      AS tamanho_docs_mb
"""))
arquivos = ["olist_orders_dataset.csv", "olist_customers_dataset.csv",
            "olist_order_items_dataset.csv", "olist_order_payments_dataset.csv"]
print("CSVs de orders + customers + items + payments:",
      round(sum(os.path.getsize(f"{DATA}/{a}") for a in arquivos) / 1e6, 1), "MB")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

documentos: 99441
{
  "order_id": "00143d0f86d6fbd9f9b38ab440ac16f5",
  "status": "delivered",
  "comprado_em": "2017-10-16 15:29:43",
  "cliente": {
    "customer_id": "2e45292979b9b2700ea09560eeb0f803",
    "pessoa": "bb8a37225e0279ac8a274c9765617eaf",
    "cidade": "goiania",
    "uf": "GO"
  },
  "itens": [
    {
      "item": 1,
      "produto": "e95ee6822b66ac6058e2e4aff656071a",
      "vendedor": "a17f621c590ea0fab3d5d883e1630ec6",
      "preco": 21.33,
      "frete": 15.1
    },
    {
      "item": 2,
      "produto": "e95ee6822b66ac6058e2e4aff656071a",
      "vendedor": "a17f621c590ea0fab3d5d883e1630ec6",
      "preco": 21.33,
      "frete": 15.1
    },
    {
      "item": 3,
      "produto": "e95ee6822b66ac6058e2e4aff656071a",
      "vendedor": "a17f621c590ea0fab3d5d883e1630ec6",
      "preco": 21.33,
      "frete": 15.1
    }
  ],
  "pagamentos": [
    {
      "seq": 1,
      "tipo": "credit_card",
      "parcelas": 1,
      "valor": 109.29
    }
  ]
}


,itens_nos_docs,itens_na_tabela,faturamento_docs,faturamento_tabela,docs_sem_itens,tamanho_docs_mb
0,112650.0,112650,13591643.7,13591643.7,775,47.0


CSVs de orders + customers + items + payments: 46.9 MB


### 4b · Embutir tem preço
No documento, os dados do cliente ficam **dentro de cada pedido**.
1. Quantas pessoas (`customer_unique_id`) aparecem com mais de um prefixo de CEP?
2. Se uma pessoa muda de endereço, quantos documentos precisam ser alterados num modelo que embute o cliente? E no modelo relacional?
3. **Hipótese a discutir, não a concluir:** por que a Olist criaria um `customer_id` novo a cada pedido? Que vantagem isso daria ao registrar o endereço de entrega?

In [5]:
sql("""
SELECT COUNT(*) AS pessoas_com_mais_de_um_cep,
       (SELECT MAX(n) FROM (SELECT customer_unique_id, COUNT(*) AS n FROM customers GROUP BY 1)) AS max_pedidos_por_pessoa
FROM (SELECT customer_unique_id FROM customers GROUP BY 1 HAVING COUNT(DISTINCT customer_zip_code_prefix) > 1)
""")

,pessoas_com_mais_de_um_cep,max_pedidos_por_pessoa
0,250,17


## 5 · Qual NoSQL para qual necessidade? (discussão, sem código)
Para cada necessidade, indique o tipo de banco mais adequado (**documento**, **chave-valor**, **colunar** ou **grafo**) e justifique em uma frase.

| Necessidade | Tipo | Justificativa |
|---|---|---|
| Consultar o status atual de um pedido pelo `order_id`, milhares de vezes por segundo | | |
| Catálogo de produtos em que cada categoria tem atributos diferentes | | |
| Recomendação "quem comprou este produto também comprou..." | | |
| Gravar bilhões de eventos de rastreamento de entrega, lidos por pedido e por data | | |

In [6]:
respostas = pd.DataFrame([
    ("Status do pedido por order_id, alta frequência", "chave-valor", "Acesso sempre pela chave, leitura e escrita muito rápidas (ex.: Redis)."),
    ("Catálogo com atributos variáveis por categoria", "documento", "Cada documento pode ter campos diferentes, sem alterar esquema (ex.: MongoDB)."),
    ("Quem comprou X também comprou Y", "grafo", "A pergunta é sobre caminhos entre nós: cliente, pedido, produto (ex.: Neo4j)."),
    ("Bilhões de eventos de rastreamento", "colunar (wide-column)", "Escrita massiva distribuída, leitura por chave de partição e ordenação por data (ex.: Cassandra)."),
], columns=["necessidade", "tipo sugerido", "justificativa"])
respostas

,necessidade,tipo sugerido,justificativa
0,"Status do pedido por order_id, alta frequência",chave-valor,"Acesso sempre pela chave, leitura e escrita mu..."
1,Catálogo com atributos variáveis por categoria,documento,"Cada documento pode ter campos diferentes, sem..."
2,Quem comprou X também comprou Y,grafo,A pergunta é sobre caminhos entre nós: cliente...
3,Bilhões de eventos de rastreamento,colunar (wide-column),"Escrita massiva distribuída, leitura por chave..."


---
# Parte 2 · Modelagem analítica: o modelo estrela da Olist
O modelo transacional da Olist é ótimo para registrar pedidos, mas responder "qual o faturamento mensal por categoria?" exige juntar `orders`, `items` e `products` a cada consulta. Na Parte 2 vamos construir um **modelo estrela**:

- **Grão da fato:** uma linha por **item vendido** (lembre do exercício 1b: cada linha de `items` é uma unidade comprada).
- **Fato:** `fato_vendas`, com as medidas `preco` e `frete`.
- **Dimensões:** `dim_produto`, `dim_vendedor`, `dim_cliente` e `dim_tempo`.

Depois da estrela, comparamos com duas outras formas de modelar para analytics: a **wide table** (tudo numa tabela só) e o **Data Vault** (hubs, links e satélites).

Os exercícios continuam a numeração: 6 a 13.

## B · Aquecimento da Parte 2: exemplos resolvidos
Três exemplos prontos com a sintaxe nova da Parte 2. Execute e leia o resultado. Eles usam a tabela `sellers` e **não** respondem aos exercícios.

### B.1 · Gerar uma chave substituta
`ROW_NUMBER() OVER (ORDER BY ...)` numera as linhas. O número vira a **chave substituta** (`sk_`) da dimensão; o identificador original (`seller_id`) fica guardado como atributo.

In [7]:
sql("""
SELECT ROW_NUMBER() OVER (ORDER BY seller_id) AS sk_vendedor,
       seller_id,
       seller_city  AS cidade,
       seller_state AS uf
FROM sellers
ORDER BY sk_vendedor
LIMIT 5
""")

,sk_vendedor,seller_id,cidade,uf
0,1,0015a82c2db000af6aaaf3ae2ecb0532,santo andre,SP
1,2,001cca7ae9ae17fb1caed9dfb1094831,cariacica,ES
2,3,001e6ad469a905060d959994f1b41e4f,sao goncalo,RJ
3,4,002100f778ceb8431b7a1020ff7ab48f,franca,SP
4,5,003554e2dce176b5555353e4f3555ac8,goiania,GO


### B.2 · Trocar nulo por um valor padrão
Uma dimensão não deve ter atributos nulos: a análise mostraria um grupo "vazio" sem nome. `COALESCE(coluna, 'valor')` substitui o nulo. Abaixo, contamos vendedores por UF e, só para demonstrar, tratamos uma UF inexistente como nula.

In [8]:
sql("""
SELECT COALESCE(NULLIF(seller_state, 'AC'), 'uf_desconhecida') AS uf,   -- NULLIF transforma 'AC' em nulo, só para o exemplo
       COUNT(*) AS vendedores
FROM sellers
WHERE seller_state IN ('AC', 'AM', 'MA', 'SP')
GROUP BY 1
ORDER BY vendedores DESC
""")

,uf,vendedores
0,SP,1849
1,MA,1
2,uf_desconhecida,1
3,AM,1


### B.3 · Gerar uma sequência de datas
A dimensão tempo tem **uma linha por dia**, inclusive dias sem venda. `generate_series` gera as datas; funções como `year()`, `month()` e `dayofweek()` extraem os atributos. Abaixo, a primeira semana de 2018.

In [9]:
sql("""
SELECT CAST(d AS DATE)   AS data,
       year(d)           AS ano,
       month(d)          AS mes,
       dayofweek(d)      AS dia_semana,   -- 0 = domingo
       strftime(d, '%Y-%m') AS ano_mes
FROM (SELECT unnest(generate_series(DATE '2018-01-01', DATE '2018-01-07', INTERVAL 1 DAY)) AS d)
""")

,data,ano,mes,dia_semana,ano_mes
0,2018-01-01,2018,1,1,2018-01
1,2018-01-02,2018,1,2,2018-01
2,2018-01-03,2018,1,3,2018-01
3,2018-01-04,2018,1,4,2018-01
4,2018-01-05,2018,1,5,2018-01
5,2018-01-06,2018,1,6,2018-01
6,2018-01-07,2018,1,0,2018-01


## 6 · O grão e o desenho da estrela
Antes de escrever SQL, decida o desenho.
1. **Grão:** uma linha da `fato_vendas` representa o quê? Quantas linhas a fato deve ter? (Dica: é o mesmo grão de `items`.)
2. **Medidas:** quais colunas de `items` são medidas (números que se somam)?
3. **Dimensões:** para cada dimensão (`produto`, `vendedor`, `cliente`, `tempo`), de qual tabela da Olist vêm os atributos?
4. **Confira com SQL** quantas linhas a fato deve ter e qual o total de `price` que ela precisa preservar.

In [10]:
sql("""
SELECT COUNT(*)                AS linhas_esperadas_na_fato,
       ROUND(SUM(price), 2)    AS preco_total_esperado,
       ROUND(SUM(freight_value), 2) AS frete_total_esperado
FROM items
""")

,linhas_esperadas_na_fato,preco_total_esperado,frete_total_esperado
0,112650,13591643.7,2256599.44


**Respostas do exercício 6**
1. **Grão:** uma linha por **unidade vendida**, o mesmo grão de `items` (Aula 2, 1b). A fato deve ter **112.650 linhas**.
2. **Medidas:** `price` e `freight_value`, ambas aditivas (podem ser somadas em qualquer dimensão). `order_item_id` não é medida: é parte do identificador.
3. **Dimensões:** produto ← `products` + `category_translation`; vendedor ← `sellers`; cliente ← `customers` (ligado pelo `customer_id` de `orders`); tempo ← `orders.order_purchase_timestamp`.
4. **Totais a preservar:** R$ 13.591.643,70 em preço e R$ 2.256.599,44 em frete.

## 7 · As dimensões
Crie as quatro dimensões com `CREATE OR REPLACE TABLE ... AS SELECT`. Cada uma deve ter uma **chave substituta** `sk_...` gerada com `ROW_NUMBER()` e o identificador original como atributo.

| Dimensão | Origem | Atributos sugeridos | Cuidado |
|---|---|---|---|
| `dim_produto` | `products` + `category_translation` | categoria, categoria em inglês, peso | 610 produtos sem categoria: use `COALESCE` |
| `dim_vendedor` | `sellers` | prefixo de CEP com 5 dígitos, cidade, UF | CEP com `lpad` (Aula 1) |
| `dim_cliente` | `customers` | pessoa (`customer_unique_id`), CEP, cidade, UF | Grão: um cliente por pedido (Aula 2) |
| `dim_tempo` | datas de `orders` | data, ano, mês, trimestre, dia da semana, ano-mês | Uma linha por dia do período, com ou sem venda |

Na `dim_tempo`, use como chave a própria data no formato `AAAAMMDD` (por exemplo, `20180115`): é a exceção usual à regra da chave sequencial, porque é estável e legível.

Ao final, conte as linhas de cada dimensão.

In [11]:
con.execute("""
CREATE OR REPLACE TABLE dim_produto AS
SELECT ROW_NUMBER() OVER (ORDER BY product_id)            AS sk_produto,
       product_id,
       COALESCE(product_category_name, 'sem_categoria')    AS categoria,
       COALESCE(t.product_category_name_english,
                CASE WHEN product_category_name IS NULL THEN 'no_category'
                     ELSE product_category_name END)        AS categoria_en,
       product_weight_g                                    AS peso_g
FROM products p
LEFT JOIN category_translation t USING (product_category_name)
""")

con.execute("""
CREATE OR REPLACE TABLE dim_vendedor AS
SELECT ROW_NUMBER() OVER (ORDER BY seller_id)               AS sk_vendedor,
       seller_id,
       lpad(CAST(seller_zip_code_prefix AS VARCHAR), 5, '0') AS cep_prefixo,
       seller_city  AS cidade,
       seller_state AS uf
FROM sellers
""")

con.execute("""
CREATE OR REPLACE TABLE dim_cliente AS
SELECT ROW_NUMBER() OVER (ORDER BY customer_id)               AS sk_cliente,
       customer_id,
       customer_unique_id                                     AS pessoa,
       lpad(CAST(customer_zip_code_prefix AS VARCHAR), 5, '0') AS cep_prefixo,
       customer_city  AS cidade,
       customer_state AS uf
FROM customers
""")

con.execute("""
CREATE OR REPLACE TABLE dim_tempo AS
SELECT CAST(strftime(d, '%Y%m%d') AS INTEGER) AS sk_tempo,
       CAST(d AS DATE)       AS data,
       year(d)               AS ano,
       month(d)              AS mes,
       quarter(d)            AS trimestre,
       dayofweek(d)          AS dia_semana,
       strftime(d, '%Y-%m')  AS ano_mes
FROM (SELECT unnest(generate_series(
        (SELECT MIN(CAST(order_purchase_timestamp AS DATE)) FROM orders),
        (SELECT MAX(CAST(order_purchase_timestamp AS DATE)) FROM orders),
        INTERVAL 1 DAY)) AS d)
""")

sql("""
SELECT 'dim_produto'  AS dimensao, COUNT(*) AS linhas FROM dim_produto
UNION ALL SELECT 'dim_vendedor', COUNT(*) FROM dim_vendedor
UNION ALL SELECT 'dim_cliente',  COUNT(*) FROM dim_cliente
UNION ALL SELECT 'dim_tempo',    COUNT(*) FROM dim_tempo
""")

,dimensao,linhas
0,dim_produto,32951
1,dim_vendedor,3095
2,dim_cliente,99441
3,dim_tempo,774


## 8 · A tabela fato
Monte a `fato_vendas` no grão definido no exercício 6: uma linha por item vendido, com
- as chaves substitutas de cada dimensão (`sk_tempo`, `sk_produto`, `sk_vendedor`, `sk_cliente`);
- `order_id` e `order_item_id` como **dimensão degenerada** (identificam o item, sem tabela própria);
- o status do pedido;
- as medidas `preco` e `frete`.

Depois, **valide a carga**: a fato tem as 112.650 linhas? Os totais de preço e frete batem com o exercício 6? Alguma chave ficou sem correspondência na dimensão?

In [12]:
con.execute("""
CREATE OR REPLACE TABLE fato_vendas AS
SELECT i.order_id,
       i.order_item_id,
       CAST(strftime(o.order_purchase_timestamp, '%Y%m%d') AS INTEGER) AS sk_tempo,
       p.sk_produto,
       v.sk_vendedor,
       c.sk_cliente,
       o.order_status   AS status_pedido,
       i.price          AS preco,
       i.freight_value  AS frete
FROM items i
JOIN orders o        USING (order_id)
JOIN dim_produto p   USING (product_id)
JOIN dim_vendedor v  USING (seller_id)
JOIN dim_cliente c   ON c.customer_id = o.customer_id
""")

sql("""
SELECT COUNT(*)                     AS linhas_na_fato,
       ROUND(SUM(preco), 2)         AS preco_total,
       ROUND(SUM(frete), 2)         AS frete_total,
       COUNT(*) FILTER (WHERE sk_tempo NOT IN (SELECT sk_tempo FROM dim_tempo)) AS sem_data_na_dim_tempo
FROM fato_vendas
""")

,linhas_na_fato,preco_total,frete_total,sem_data_na_dim_tempo
0,112650,13591643.7,2256599.44,0


## 9 · A mesma pergunta nos dois modelos
Pergunta de negócio: **qual o faturamento mensal por categoria, só de pedidos entregues?**
1. Responda no **modelo estrela** (`fato_vendas` + `dim_tempo` + `dim_produto`).
2. Responda no **modelo transacional** (`items` + `orders` + `products`).
3. Compare: os resultados são iguais? Qual consulta foi mais fácil de escrever e de ler? Qual depende de lembrar regras (como tratar categoria nula)?
4. Quais as 3 categorias com maior faturamento entregue?

In [13]:
estrela = sql("""
SELECT t.ano_mes, p.categoria, ROUND(SUM(f.preco), 2) AS faturamento
FROM fato_vendas f
JOIN dim_tempo t   USING (sk_tempo)
JOIN dim_produto p USING (sk_produto)
WHERE f.status_pedido = 'delivered'
GROUP BY 1, 2
ORDER BY 1, 2
""")

transacional = sql("""
SELECT strftime(o.order_purchase_timestamp, '%Y-%m')        AS ano_mes,
       COALESCE(pr.product_category_name, 'sem_categoria')  AS categoria,
       ROUND(SUM(i.price), 2)                               AS faturamento
FROM items i
JOIN orders o    USING (order_id)
JOIN products pr USING (product_id)
WHERE o.order_status = 'delivered'
GROUP BY 1, 2
ORDER BY 1, 2
""")

print("Linhas na resposta:", len(estrela), "| Resultados idênticos:", estrela.equals(transacional))
sql("""
SELECT p.categoria, ROUND(SUM(f.preco), 2) AS faturamento_entregue, COUNT(*) AS itens
FROM fato_vendas f JOIN dim_produto p USING (sk_produto)
WHERE f.status_pedido = 'delivered'
GROUP BY 1 ORDER BY 2 DESC LIMIT 3
""")

Linhas na resposta: 1273 | Resultados idênticos: True


,categoria,faturamento_entregue,itens
0,beleza_saude,1233131.72,9465
1,relogios_presentes,1166176.98,5859
2,cama_mesa_banho,1023434.76,10953


**Comparação:** os resultados são **idênticos** (1.273 linhas, `equals = True`). A diferença está em onde mora a regra de negócio: na estrela, o tratamento da categoria nula (`sem_categoria`) foi feito **uma vez**, na `dim_produto`, e toda consulta herda. No transacional, cada consulta precisa repetir o `COALESCE`; quem esquecer gera um grupo nulo e um resultado diferente sem nenhum erro. A estrela também é mais legível: a consulta fala de "categoria" e "ano_mes", e não de `strftime` sobre timestamp.

**Top 3 entregue:** beleza_saude (R$ 1.233.131,72), relogios_presentes (R$ 1.166.176,98) e cama_mesa_banho (R$ 1.023.434,76). Cama_mesa_banho é a que mais vende em unidades (10.953) e só a terceira em receita: ticket médio menor.

## 10 · De estrela para snowflake
No **snowflake**, uma dimensão é normalizada: a categoria sai de `dim_produto` e vai para uma tabela própria.
1. Crie `dim_categoria` (chave substituta, categoria, categoria em inglês), com uma linha por categoria.
2. Crie `dim_produto_sf`, igual à `dim_produto`, mas com `sk_categoria` no lugar dos nomes da categoria.
3. Reescreva a consulta do top 3 do exercício 9 no snowflake. Quantos JOINs ela precisa agora?
4. **Discussão:** o que se ganha e o que se perde em relação à estrela?

In [14]:
con.execute("""
CREATE OR REPLACE TABLE dim_categoria AS
SELECT ROW_NUMBER() OVER (ORDER BY categoria) AS sk_categoria,
       categoria,
       any_value(categoria_en)                AS categoria_en
FROM dim_produto
GROUP BY categoria
""")
con.execute("""
CREATE OR REPLACE TABLE dim_produto_sf AS
SELECT p.sk_produto, p.product_id, c.sk_categoria, p.peso_g
FROM dim_produto p JOIN dim_categoria c USING (categoria)
""")
display(sql("SELECT (SELECT COUNT(*) FROM dim_categoria) AS categorias, (SELECT COUNT(*) FROM dim_produto_sf) AS produtos"))

sql("""
SELECT c.categoria, ROUND(SUM(f.preco), 2) AS faturamento_entregue, COUNT(*) AS itens
FROM fato_vendas f
JOIN dim_produto_sf p USING (sk_produto)
JOIN dim_categoria c  USING (sk_categoria)
WHERE f.status_pedido = 'delivered'
GROUP BY 1 ORDER BY 2 DESC LIMIT 3
""")

,categorias,produtos
0,74,32951


,categoria,faturamento_entregue,itens
0,beleza_saude,1233131.72,9465
1,relogios_presentes,1166176.98,5859
2,cama_mesa_banho,1023434.76,10953


**JOINs:** a consulta do top 3 passa de **1 JOIN** na estrela (fato → produto) para **2** no snowflake (fato → produto → categoria).

**Ganha-se:** o nome de cada categoria fica guardado uma vez só (74 linhas, em vez de repetido em 32.951 produtos); corrigir ou traduzir uma categoria é atualizar uma linha; a hierarquia fica explícita.
**Perde-se:** um JOIN a mais em toda consulta por categoria e um modelo mais difícil de entender para quem usa ferramentas de BI. Como as dimensões são pequenas perto da fato, a economia de espaço é desprezível. Por isso a estrela costuma ser o padrão, e o snowflake fica para hierarquias grandes ou compartilhadas.

## 11 · Quando a dimensão muda (discussão, sem código)
Na Aula 2 vimos que **250 pessoas** (`customer_unique_id`) aparecem com mais de um prefixo de CEP.

Suponha uma dimensão de cliente com **uma linha por pessoa** (e não por pedido, como a nossa `dim_cliente`). Quando a pessoa muda de cidade:

| Estratégia | O que acontece na dimensão | Consequência para "vendas por estado nos últimos 3 anos" |
|---|---|---|
| **SCD tipo 1:** sobrescrever | A linha da pessoa é atualizada com a cidade e a UF novas. O endereço antigo é perdido | **Distorce o histórico:** todas as compras antigas passam a ser contadas no estado novo, como se ela sempre tivesse morado lá |
| **SCD tipo 2:** nova linha com nova chave substituta e datas de validade | A linha antiga é encerrada (data de fim, `atual = false`) e entra uma linha nova, com nova `sk_cliente` e data de início | **Preserva o histórico:** cada venda aponta para a versão do cliente válida na data da compra, e as compras antigas continuam no estado antigo. Em troca, a dimensão cresce e a carga da fato precisa buscar a versão vigente |

**Item 2:** a nossa `dim_cliente`, com uma linha por **cliente no pedido**, já se comporta como **SCD tipo 2** sem nenhum esforço: cada pedido tem a própria linha, com o endereço daquele momento, e nenhuma compra antiga é reescrita. O que ela não oferece é uma visão do "endereço atual" da pessoa: para isso seria preciso pegar a linha do pedido mais recente de cada `pessoa`.

## 12 · Wide table: tudo numa tabela só
A **wide table** junta a fato e os atributos das dimensões numa única tabela larga. Quem consulta não faz nenhum JOIN.
1. Crie `wide_vendas` a partir da estrela: uma linha por item vendido, com data, ano-mês, status, categoria, peso, cidade e UF do vendedor, pessoa, cidade e UF do cliente, preço e frete.
2. Confira as linhas e o total de preço.
3. Refaça o top 3 do exercício 9 sem nenhum JOIN.
4. **Discussão:** o nome de uma categoria aparece em quantas linhas? O que acontece se ele precisar ser corrigido?

In [15]:
con.execute("""
CREATE OR REPLACE TABLE wide_vendas AS
SELECT f.order_id, f.order_item_id,
       t.data AS data_compra, t.ano_mes, t.ano,
       f.status_pedido,
       p.categoria, p.peso_g,
       v.cidade AS vendedor_cidade, v.uf AS vendedor_uf,
       c.pessoa, c.cidade AS cliente_cidade, c.uf AS cliente_uf,
       f.preco, f.frete
FROM fato_vendas f
JOIN dim_tempo t    USING (sk_tempo)
JOIN dim_produto p  USING (sk_produto)
JOIN dim_vendedor v USING (sk_vendedor)
JOIN dim_cliente c  USING (sk_cliente)
""")
display(sql("SELECT COUNT(*) AS linhas, ROUND(SUM(preco), 2) AS preco_total FROM wide_vendas"))
print("Colunas:", len(sql("SELECT * FROM wide_vendas LIMIT 0").columns))

display(sql("""
SELECT categoria, ROUND(SUM(preco), 2) AS faturamento_entregue
FROM wide_vendas
WHERE status_pedido = 'delivered'
GROUP BY 1 ORDER BY 2 DESC LIMIT 3
"""))
sql("SELECT categoria, COUNT(*) AS linhas_com_este_nome FROM wide_vendas GROUP BY 1 ORDER BY 2 DESC LIMIT 3")

,linhas,preco_total
0,112650,13591643.7


Colunas: 15


,categoria,faturamento_entregue
0,beleza_saude,1233131.72
1,relogios_presentes,1166176.98
2,cama_mesa_banho,1023434.76


,categoria,linhas_com_este_nome
0,cama_mesa_banho,11115
1,beleza_saude,9670
2,esporte_lazer,8641


**Redundância:** o nome `cama_mesa_banho` aparece em **11.115 linhas**, `beleza_saude` em 9.670. Corrigir o nome de uma categoria exige atualizar milhares de linhas, e qualquer falha deixa a mesma categoria escrita de dois jeitos (a anomalia de atualização da Aula 2). Por isso a wide table não é a fonte da verdade: ela é **derivada** da estrela e reconstruída a cada carga. O ganho é o consumo: nenhum JOIN, ideal para ferramentas de BI, planilhas e modelos de machine learning.

## 13 · Data Vault: hubs, links e satélites
O **Data Vault** não é feito para consulta direta: é um modelo de **integração**, que guarda tudo o que chega das fontes, com histórico e rastreabilidade de origem.
- **Hub:** uma linha por chave de negócio (pedido, produto, vendedor), com uma chave *hash*, a data de carga e a fonte.
- **Link:** uma linha por relacionamento entre hubs (este pedido tem este produto deste vendedor).
- **Satélite:** os atributos descritivos de um hub ou link, com a data de carga, para guardar versões.

1. Crie `hub_pedido`, `hub_produto` e `hub_vendedor` com `md5()` do identificador como chave.
2. Crie `link_item`, com uma linha por combinação distinta de pedido, produto e vendedor.
3. Crie `sat_produto` (categoria, peso), `sat_pedido` (status, data) e `sat_item` (preço, frete, com `order_item_id`).
4. Conte as linhas de cada tabela e refaça o top 3 do exercício 9 a partir do Data Vault. Quantos JOINs foram necessários?

In [16]:
con.execute("CREATE OR REPLACE TABLE hub_pedido   AS SELECT md5(order_id)   AS hk_pedido,   order_id,   now() AS load_ts, 'olist.orders'   AS fonte FROM orders")
con.execute("CREATE OR REPLACE TABLE hub_produto  AS SELECT md5(product_id) AS hk_produto,  product_id, now() AS load_ts, 'olist.products' AS fonte FROM products")
con.execute("CREATE OR REPLACE TABLE hub_vendedor AS SELECT md5(seller_id)  AS hk_vendedor, seller_id,  now() AS load_ts, 'olist.sellers'  AS fonte FROM sellers")
con.execute("""
CREATE OR REPLACE TABLE link_item AS
SELECT DISTINCT md5(order_id || '|' || product_id || '|' || seller_id) AS hk_link,
       md5(order_id) AS hk_pedido, md5(product_id) AS hk_produto, md5(seller_id) AS hk_vendedor,
       now() AS load_ts, 'olist.items' AS fonte
FROM items
""")
con.execute("""
CREATE OR REPLACE TABLE sat_produto AS
SELECT md5(product_id) AS hk_produto, now() AS load_ts,
       COALESCE(product_category_name, 'sem_categoria') AS categoria, product_weight_g AS peso_g
FROM products
""")
con.execute("CREATE OR REPLACE TABLE sat_pedido AS SELECT md5(order_id) AS hk_pedido, now() AS load_ts, order_status AS status, order_purchase_timestamp AS comprado_em FROM orders")
con.execute("""
CREATE OR REPLACE TABLE sat_item AS
SELECT md5(order_id || '|' || product_id || '|' || seller_id) AS hk_link, order_item_id,
       now() AS load_ts, price AS preco, freight_value AS frete
FROM items
""")
display(sql("""
SELECT 'hub_pedido' AS tabela, COUNT(*) AS linhas FROM hub_pedido
UNION ALL SELECT 'hub_produto',  COUNT(*) FROM hub_produto
UNION ALL SELECT 'hub_vendedor', COUNT(*) FROM hub_vendedor
UNION ALL SELECT 'link_item',    COUNT(*) FROM link_item
UNION ALL SELECT 'sat_produto',  COUNT(*) FROM sat_produto
UNION ALL SELECT 'sat_pedido',   COUNT(*) FROM sat_pedido
UNION ALL SELECT 'sat_item',     COUNT(*) FROM sat_item
"""))
sql("""
SELECT sp.categoria, ROUND(SUM(si.preco), 2) AS faturamento_entregue
FROM sat_item si
JOIN link_item l     USING (hk_link)
JOIN sat_produto sp  USING (hk_produto)
JOIN sat_pedido spe  USING (hk_pedido)
WHERE spe.status = 'delivered'
GROUP BY 1 ORDER BY 2 DESC LIMIT 3
""")

,tabela,linhas
0,hub_pedido,99441
1,hub_produto,32951
2,hub_vendedor,3095
3,link_item,102425
4,sat_produto,32951
5,sat_pedido,99441
6,sat_item,112650


,categoria,faturamento_entregue
0,beleza_saude,1233131.72
1,relogios_presentes,1166176.98
2,cama_mesa_banho,1023434.76


**Contagens:** hubs com 99.441 pedidos, 32.951 produtos e 3.095 vendedores; `link_item` com **102.425** combinações pedido-produto-vendedor, menos que os 112.650 itens porque unidades repetidas do mesmo produto no mesmo pedido viram um único relacionamento (10.225 = 17.313 − 7.088, o mesmo achado do 1b da Aula 2). O `sat_item` recupera as 112.650 unidades pelo `order_item_id`.

**JOINs:** o top 3 precisou de **3 JOINs** (satélite do item → link → satélite do produto e satélite do pedido), contra 1 na estrela e nenhum na wide table. O resultado é o mesmo. O Data Vault não foi feito para consulta direta: ele serve para **integrar** fontes com histórico e rastreabilidade (cada linha diz de onde veio e quando foi carregada). Na prática, a estrela é construída **a partir** dele.

## Modelo estrela da equipe (entregue ao fim da prática)
Registre o modelo construído. Uma linha por tabela.

| Tabela | Tipo (fato ou dimensão) | Grão (uma linha representa...) | Chave | Linhas |
|---|---|---|---|---|
| fato_vendas | Fato | Uma unidade vendida | `order_id` + `order_item_id` (dimensão degenerada); FKs `sk_tempo`, `sk_produto`, `sk_vendedor`, `sk_cliente` | 112.650 |
| dim_produto | Dimensão | Um produto do catálogo | `sk_produto` | 32.951 |
| dim_vendedor | Dimensão | Um vendedor | `sk_vendedor` | 3.095 |
| dim_cliente | Dimensão | O cliente num pedido (comportamento de SCD tipo 2) | `sk_cliente` | 99.441 |
| dim_tempo | Dimensão | Um dia, de 04/09/2016 a 17/10/2018, com ou sem venda | `sk_tempo` (AAAAMMDD) | 774 |
| dim_categoria + dim_produto_sf | Dimensão (snowflake) | Uma categoria / um produto apontando para a categoria | `sk_categoria` / `sk_produto` | 74 / 32.951 |
| wide_vendas | Tabela larga derivada da estrela | Uma unidade vendida com todos os atributos | `order_id` + `order_item_id` | 112.650 (15 colunas) |
| Data Vault (hubs, link, satélites) | Modelo de integração | Hub: uma chave de negócio; link: um relacionamento pedido-produto-vendedor; satélite: atributos de uma carga | Hash `md5` da chave de negócio | Hubs 99.441 / 32.951 / 3.095; link 102.425; satélites 32.951 / 99.441 / 112.650 |